# 01 — Load & Inspect LIAR

**Goal:** load LIAR train/valid/test TSV files and do basic inspection:
- dataset shapes
- label distribution
- example statements
- simple text stats (length)

**Expected data path (edit if needed):**
- `data/liar/train.tsv`
- `data/liar/valid.tsv`
- `data/liar/test.tsv`


## 0) Setup (edit path if needed)

- Change `DATA_DIR` if your LIAR TSV files are stored elsewhere.


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('data/liar')  # change if needed
train_path = DATA_DIR / 'train.tsv'
valid_path = DATA_DIR / 'valid.tsv'
test_path  = DATA_DIR / 'test.tsv'

cols = [
    'id','label','statement','subject','speaker','speaker_job','state','party',
    'barely_true_counts','false_counts','half_true_counts','mostly_true_counts',
    'pants_on_fire_counts','context'
]

def load_tsv(p: Path) -> pd.DataFrame:
    if not p.exists():
        raise FileNotFoundError(
            f'File not found: {p}.\n'
            f'Please upload LIAR TSVs to {DATA_DIR}/ or change DATA_DIR.'
        )
    return pd.read_csv(p, sep='\t', header=None, names=cols)

train = load_tsv(train_path)
valid = load_tsv(valid_path)
test  = load_tsv(test_path)

print('Loaded LIAR TSVs successfully.')


## 1) Basic shapes

Print number of rows/columns for each split.

In [ ]:
print('Shapes:')
print('  train:', train.shape)
print('  valid:', valid.shape)
print('  test :', test.shape)


## 2) Label distribution

Show label counts for each split.

In [ ]:
def show_label_counts(df: pd.DataFrame, name: str):
    print(f'\n{name} label counts:')
    print(df['label'].value_counts())

show_label_counts(train, 'Train')
show_label_counts(valid, 'Valid')
show_label_counts(test,  'Test')


## 3) Example statements

Print a few examples to understand what the text looks like.

In [ ]:
print('Sample statements (train):')
for i, s in enumerate(train['statement'].head(5).astype(str).tolist(), 1):
    print(f'{i}. {s}')


## 4) Basic text quality checks

- Missing/empty statements
- Simple length statistics


In [ ]:
def text_stats(df: pd.DataFrame, name: str):
    s = df['statement'].astype(str)
    empty = (s.str.strip() == '').sum()
    lengths = s.str.len()
    print(f'\n{name} text stats:')
    print('  empty statements:', empty)
    print('  length (chars) min/median/mean/max:',
          int(lengths.min()), int(lengths.median()), float(lengths.mean()), int(lengths.max()))

text_stats(train, 'Train')
text_stats(valid, 'Valid')
text_stats(test,  'Test')


## 5) Optional: save a short overview file

This creates a small markdown summary under `results/` in your runtime. You can upload it back to GitHub.

In [ ]:
from datetime import datetime

def label_counts_str(df: pd.DataFrame) -> str:
    vc = df['label'].value_counts()
    return '\n'.join([f'- {k}: {int(v)}' for k, v in vc.items()])

overview = []
overview.append('# LIAR Dataset Overview')
overview.append('')
overview.append(f'- Date: {datetime.now().strftime("%Y-%m-%d %H:%M") }')
overview.append(f'- Data dir: {DATA_DIR}')
overview.append('')
overview.append('## Shapes')
overview.append(f'- Train: {train.shape}')
overview.append(f'- Valid: {valid.shape}')
overview.append(f'- Test:  {test.shape}')
overview.append('')
overview.append('## Label counts (Train)')
overview.append(label_counts_str(train))
overview.append('')
overview.append('## Label counts (Test)')
overview.append(label_counts_str(test))
overview.append('')
overview.append('## Sample statements (Train)')
for i, s in enumerate(train['statement'].head(3).astype(str).tolist(), 1):
    overview.append(f'- {i}. {s}')

out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'liar_overview.md'
out_path.write_text('\n'.join(overview), encoding='utf-8')

print('Saved:', out_path.resolve())
